In [1]:
%pip install -q youtube-transcript-api langchain-community langchain-text-splitters langchain-cohere langchain-groq faiss-cpu python-dotenv


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is not set. Set it in your terminal before starting Jupyter.")
if not os.getenv("COHERE_API_KEY"):
    raise ValueError("COHERE_API_KEY is not set. Set it in your terminal before starting Jupyter.")

/Users/shubham/Desktop/lanchain_models/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/j6/wvsdyk8x5dn18c2cv28bf0nc0000gn/T/ipykernel_76206/3040248404.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 1a - Indexing (Document Ingestion)

In [6]:
video_id = "nDLb8_wgX50"  # only the ID, not the full URL
try:
    # Current youtube-transcript-api interface
    transcript_data = YouTubeTranscriptApi().fetch(video_id, languages=["en"])
    transcript = " ".join(snippet.text for snippet in transcript_data)
    print(transcript)
except TranscriptsDisabled:
    print("No captions available for this video.")

ANDREW HUBERMAN: Welcome
to the Huberman Lab podcast where we discuss science
and science-based tools for everyday life. [MUSIC PLAYING] I'm Andrew Huberman,
and I'm a professor of neurobiology
and ophthalmology at Stanford School of Medicine. My guest today is David Goggins. David Goggins is a
retired Navy SEAL who served in Iraq and Afghanistan. He's also a highly accomplished
ultramarathon runner. For those of you
that don't know, ultramarathons are
distances longer than 26 miles and, in
David's case, often longer than 200 miles. For his achievements
in athletics, he has been inducted into
the International Sports Hall of Fame. He also held a Guinness World
Record for the most pull-ups completed in 24 hours. I should mention that not only
was David a decorated Navy SEAL, but he also graduated
from Army Ranger School. David is also a highly
successful writer, having authored two books, the
first entitled "Can't Hurt Me" and the second entitled "Never
Finished," both of which are best

In [8]:
print(transcript_data)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='ANDREW HUBERMAN: Welcome\nto the Huberman Lab podcast', start=0.0, duration=2.167), FetchedTranscriptSnippet(text='where we discuss science\nand science-based tools', start=2.167, duration=2.653), FetchedTranscriptSnippet(text='for everyday life.', start=4.82, duration=0.78), FetchedTranscriptSnippet(text='[MUSIC PLAYING]', start=5.6, duration=2.96), FetchedTranscriptSnippet(text="I'm Andrew Huberman,\nand I'm a professor", start=9.06, duration=1.95), FetchedTranscriptSnippet(text='of neurobiology\nand ophthalmology', start=11.01, duration=2.25), FetchedTranscriptSnippet(text='at Stanford School of Medicine.', start=13.26, duration=1.98), FetchedTranscriptSnippet(text='My guest today is David Goggins.', start=15.24, duration=2.43), FetchedTranscriptSnippet(text='David Goggins is a\nretired Navy SEAL who', start=17.67, duration=2.58), FetchedTranscriptSnippet(text='served in Iraq and Afghanistan.', start=20.25, duration=2.23), F

## Step 1b - Indexing (Text Splitting)

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [10]:
len(chunks)

183

In [11]:
# Preview a chunk if one exists
if chunks:
    print(chunks[min(100, len(chunks) - 1)].page_content)
else:
    print("No chunks were created. Check that the transcript was retrieved.")

should be for a lot of people. But there'll never be balance
because confidence is something that you're constantly-- confidence and belief
you're building every day. And so something's got to give. And I'm willing to
forego a lot of things to have that because
I know if you want to give
somebody kryptonite, take that shit away from them. So yeah, I don't
sleep sometimes. And sometimes I don't
eat the right way. And sometimes I don't do this
and do that and whatever, man. But you put me in a room
of 10,000 people any time of the day, and I'll
walk in there thinking I'm the baddest
motherfucker in here because I know what it
took to be on this stage. And a lot of people
would not do that. So that's what it takes. ANDREW HUBERMAN:
There's a question I've been wanting to ask
you since we started. And I thought about
coming in here. And I've been thinking about
in the weeks ahead of this. And I'm going to just come


## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [12]:
embeddings = CohereEmbeddings(model="embed-english-v3.0")
vector_store = FAISS.from_documents(chunks, embeddings)

In [13]:
vector_store.index_to_docstore_id

{0: '91a5057b-da9f-4227-8886-f17d2a4cbd8b',
 1: 'cd8e2bdc-9b76-4b7a-805a-b51d5707ac6e',
 2: '5b506a67-4517-4964-bdab-5810314b80ac',
 3: '7b75607d-1585-4ec9-a631-c187938f69f7',
 4: 'e568dd66-af53-4a95-aa60-0e380a6026f1',
 5: '97a38da1-f93a-47ea-81b3-8c843fa06657',
 6: 'a3e24216-88cb-4763-8fa3-594c51ba7bb8',
 7: '142ff3f0-d5d6-4dde-bf56-e860dbda6221',
 8: 'dc1c73f7-9797-4a95-ac63-b256f61cbf3d',
 9: 'a462394d-1444-43be-b40d-d074006325b4',
 10: 'fafd985d-3ab9-4e15-8d45-5d935dd2aab2',
 11: '111852e4-801c-405d-81a0-b254c3914150',
 12: 'dedd6135-a0d4-4447-bd6d-835fa85d6361',
 13: '5962f429-bf6c-485b-b19a-a9c2ac6487eb',
 14: '243ef6a1-3498-4d4f-83e3-3562e4d9c42d',
 15: 'ca39c697-58e6-46ac-8da2-94a41f4acfb2',
 16: '9e7d6882-941b-43c6-a0fa-5529445f2dc1',
 17: 'a69dbbe0-a88c-438a-89da-d217fd106b62',
 18: 'd4beda7c-a0b5-4f8b-9737-0684b49131b2',
 19: '1c6444a8-08b2-4dde-af82-d126fd8c2778',
 20: '1c3d921c-4fd4-436c-9843-04b766b87dbd',
 21: '2ea174d0-91c0-4a18-b921-65bec5ad7971',
 22: '1e9fc129-bd25-

In [14]:
vector_store.get_by_ids(['a90dfebf-9731-485f-b446-36b53e381b11'])

[Document(id='a90dfebf-9731-485f-b446-36b53e381b11', metadata={}, page_content="you developed a skill. But it's a perishable skill. And you clearly\nlive in the process of opening those cupboards,\nreopening those cupboards, trying to spit shine\nthose cupboards, understanding that they're\nnever, ever really done, but that you can gain ground\non them, that you can win day after day after day. And you really shared a\nlot of concrete things that I know people\nare going to be able to apply if they choose. And I agree with you. I think most people\nwill be like, whoa. That was a lot. It's heavy. I think I want to just kind of\nbake myself in Netflix and Chex Mix instead. But there's also the\nreality that there are men and women, boys and\ngirls who hear that and go, OK, and start cracking\nthe cupboards open. And I just know\nthat for myself, I'm extremely grateful that\nyou're willing to put it all out there. You're so brutally honest,\nso brutally authentic. That word authenticity\n

## Step 2 - Retrieval

In [16]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [17]:
retriever

VectorStoreRetriever(tags=['FAISS', 'CohereEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x127edda90>, search_kwargs={'k': 4})

In [20]:
retriever.invoke('Who is david')

[Document(id='cd8e2bdc-9b76-4b7a-805a-b51d5707ac6e', metadata={}, page_content='from Army Ranger School. David is also a highly\nsuccessful writer, having authored two books, the\nfirst entitled "Can\'t Hurt Me" and the second entitled "Never\nFinished," both of which are best sellers. David\'s books cover\nmany topics, including his autobiographical\ndescription of what can only be described as an\nincredibly challenging child and young adulthood. His home was abusive. His school environment\nwas abusive. He essentially had no positive\nresources directed his way and. In his 20s, he found himself\nto be obese, that is, more than 300\npounds working a job he despised for minimal pay. And it was at that point that\nDavid began an inner dialogue that forced him to explore\nthe demons born out of his childhood but also the\nposition that he found himself in as a young man and\nthen began the journey to navigate that dialogue and\ntransform himself into the Navy SEAL, the ultramarathon\nru

## Step 3 - Augmentation

In [21]:
# Groq-hosted chat model
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2)

In [22]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [23]:
question          = "is the topic of mental toughness discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [24]:
retrieved_docs

[Document(id='376e8052-8de4-4555-911c-42521240d198', metadata={}, page_content="like an RPM or high. [REVVING SOUND]\nAnyway, that's me. But yeah, I think\ncertain things over time, I feel like talent or gifts or\nwhatever you want to call them, but there are many\nthings that are exceedingly difficult for me. And I have learned\nfrom your example. I know that you are very\nboth humble and very clear that you don't have--\nyou say, you're not going to get it\nby examining you. But I think the way\nyou're sharing today and the way you've shared\nit on other podcasts before, there are pieces\nthat really help people feel into\nthe process of what you're talking about today. We're elaborating on it, I\nthink, a lot, this notion of being haunted and the stick. I mean, of course, of course,\nnow it makes so much sense why you don't want to talk about\nsleep or rest or recovery because that's-- sure, that's important. I've heard you say,\nyes, you sleep. Yes, you eat. Yes, you hydrate. Yes, 

In [25]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"like an RPM or high. [REVVING SOUND]\nAnyway, that's me. But yeah, I think\ncertain things over time, I feel like talent or gifts or\nwhatever you want to call them, but there are many\nthings that are exceedingly difficult for me. And I have learned\nfrom your example. I know that you are very\nboth humble and very clear that you don't have--\nyou say, you're not going to get it\nby examining you. But I think the way\nyou're sharing today and the way you've shared\nit on other podcasts before, there are pieces\nthat really help people feel into\nthe process of what you're talking about today. We're elaborating on it, I\nthink, a lot, this notion of being haunted and the stick. I mean, of course, of course,\nnow it makes so much sense why you don't want to talk about\nsleep or rest or recovery because that's-- sure, that's important. I've heard you say,\nyes, you sleep. Yes, you eat. Yes, you hydrate. Yes, you will stretch\nyour psoas or whatever. But it's funny how that\n\naspect of 

In [26]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [27]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      like an RPM or high. [REVVING SOUND]\nAnyway, that's me. But yeah, I think\ncertain things over time, I feel like talent or gifts or\nwhatever you want to call them, but there are many\nthings that are exceedingly difficult for me. And I have learned\nfrom your example. I know that you are very\nboth humble and very clear that you don't have--\nyou say, you're not going to get it\nby examining you. But I think the way\nyou're sharing today and the way you've shared\nit on other podcasts before, there are pieces\nthat really help people feel into\nthe process of what you're talking about today. We're elaborating on it, I\nthink, a lot, this notion of being haunted and the stick. I mean, of course, of course,\nnow it makes so much sense why you don't want to talk about\nsleep or rest or recovery beca

## Step 4 - Generation

In [28]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes. The conversation is centered on mental toughness.  

**What was discussed**

- **Willpower and the brain** – Andrew Huberman explains that everyone has two anterior mid‑cingulate cortices, a brain region linked to willpower. Strengthening it isn’t about doing activities you already enjoy; you have to engage in truly uncomfortable, effort‑heavy tasks.

- **The “friction” or “suck”** – The speakers describe the unpleasant, haunted feeling that comes with hard work (the “stick,” “haunt,” “suck”). They argue that this discomfort is the critical lever that builds mental toughness; merely putting in effort isn’t enough.

- **David Goggins as a model** – Goggins is referenced as a verb for pushing through extreme mental and physical hardship. His approach exemplifies embracing the painful, dungeon‑like experience rather than seeking motivation from heroic, cinematic narratives.

- **Practical examples** – They mention training methods such as sets to failure, forced reps, and other “hard

## Building a Chain

In [30]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [31]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [32]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [33]:
parallel_chain.invoke('who is David')

{'context': 'from Army Ranger School. David is also a highly\nsuccessful writer, having authored two books, the\nfirst entitled "Can\'t Hurt Me" and the second entitled "Never\nFinished," both of which are best sellers. David\'s books cover\nmany topics, including his autobiographical\ndescription of what can only be described as an\nincredibly challenging child and young adulthood. His home was abusive. His school environment\nwas abusive. He essentially had no positive\nresources directed his way and. In his 20s, he found himself\nto be obese, that is, more than 300\npounds working a job he despised for minimal pay. And it was at that point that\nDavid began an inner dialogue that forced him to explore\nthe demons born out of his childhood but also the\nposition that he found himself in as a young man and\nthen began the journey to navigate that dialogue and\ntransform himself into the Navy SEAL, the ultramarathon\nrunner, the bestselling author, and the extraordinarily\n\nthen began

In [34]:
parser = StrOutputParser()

In [35]:
main_chain = parallel_chain | prompt | llm | parser

In [36]:
main_chain.invoke('Can you summarize the video')

'**Summary of the video (based on the transcript):**\n\nThe segment opens with a raw, impassioned monologue that rejects “life‑hack” shortcuts and emphasizes that real growth comes only from relentless, gritty effort—doing the hard work over and over, even when it feels miserable. The speaker talks about confronting painful personal history (e.g., an abusive father), acknowledging that those wounds still affect him, and insisting that true change requires reliving and owning every detail of that pain rather than glossing over it. He frames life as a continuous test, one that catches us off‑guard when we think we’re comfortable, and argues that most people fail because they don’t understand their purpose.\n\nThe tone then shifts to a conversation between **Andrew Huberman** and **David Goggins**. Huberman introduces a neuroscience tidbit (the name of a brain structure is cut off in the transcript) and asks Goggins about a side of him that people don’t usually see. Goggins explains that,